# Golden window - which X range makes a batch fail QC, with statistical evidence

**Golden** = every Y inside `config.GOLDEN_SPEC` (Kappa 8-10, Viscosity 700-800, confirmed 2026-10-28).
Each X is tested for each **fail mode**, because one X can push two Y the same or opposite ways.

| Step | What | Why it is evidence |
|---|---|---|
| 1 | Fail rate per quintile of X (Wilson 95% CI) | shows *where* the fail rate jumps |
| 2 | Best cut + risk ratio + **permutation p** (outcome shuffled inside each month) | the p already pays for searching the cut, and the season cannot create it |
| 3 | **Mantel-Haenszel by month** + months that agree | same rule inside every month -> not a season effect |
| 4 | Cut from the early months, checked on the **later months** | the rule works on data it never saw |
| 5 | **Benjamini-Hochberg** q over all X | many X tested -> control false finds |
| 6 | Share on the bad side now, fail rate if moved; combined window check | what to control and what it is worth |

**Grade A** = 5 + 3 + 4 + every month agrees · **B** = 5 + (3 or 4) · **C** = 5 only · **-** = not significant.
Observational data: grade A is strong enough to propose a **trial**, not to prove cause.

Edit X in `config.FEATURES` / `config.GOLDEN_EXTRA`, the combined window in `config.GOLDEN_WINDOW_RULES`. No PI access.

In [ ]:
import importlib
import pandas as pd

import config, data, golden, plots
for mod in (config, data, golden, plots):
    importlib.reload(mod)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 40)

## Step 0 - Data and why batches fail
A batch can fail more than one way, so the modes add up to more than `fail`.

In [ ]:
df = golden.load_golden()
overall, by_month = golden.fail_modes(df)
display(overall)
display(by_month)

## Outliers of X
Setting: `config.GOLDEN_OUTLIER` = `None` (keep all, default) / `"2sd"` / `"iqr3"`, removed **one X at a time** and per digester.
- Table 1: how many values each method would remove (high `skew` -> 2 SD is a poor rule, it only cuts the long side).
- Table 2 (**sensitivity**): the same threshold test with no / 2 SD / 3 IQR removal for the window X. If the cut, fail rates and months barely move, the result does not come from outliers.

In [ ]:
print("GOLDEN_OUTLIER =", config.GOLDEN_OUTLIER, "| by digester:", config.GOLDEN_OUTLIER_BY_DIGESTER)
display(golden.outlier_report(df))
for col, _, _ in config.GOLDEN_WINDOW_RULES:
    print(f"Sensitivity - {golden.golden_features(df).get(col, col)} (Fail QC any)")
    display(golden.sensitivity(df, col, "fail"))

## Steps 1-5 - Screen every X for every fail mode
How to read one row of the summary:
- **Bad side** `<= 64.3` -> batches with X at or below 64.3 fail more. *Cut 90% range* = how much the cut moves in a week-block bootstrap.
- **Fail % at bad side / Fail % at good side** -> both are FAIL rates: on the side that fails more (bad) and on the other side (good). **RR** = how many times more likely on the bad side.
- **p perm (q)** -> chance of a cut this strong by luck *within the same month*; q = after BH.
- **MH RR by month / Months agree** -> the effect after removing the season, and in how many months it points the same way.
- **Future fail % at bad / good side, Future p** -> cut found on the early months, fail rates in the later months.
- **Fail now -> if moved** -> fail rate if every bad-side batch moved to the good side (month-adjusted, upper estimate).

In [ ]:
results = {}
for event in golden.OUTCOMES:
    res, _ = golden.screen(df, event, progress=False)
    results[event] = res
    print()
    print(f"===== {golden.OUTCOMES[event]}  (overall {df[event].mean():.0%}) =====")
    display(golden.summary(res))

### Step 1 picture - fail rate per quintile of the top X
Orange = bins on the bad side of the cut. Error bars = Wilson 95% CI. Bars that overlap the dashed line are not different from average.

In [ ]:
for event in ["fail", "kappa_high", "visc_high"]:
    plots.fail_curves(df, results[event], event)

## Step 6 - Operating window
From the grade A/B rows: a bad side `<=` gives a **lower limit** (keep X above), `>` gives an **upper limit**.
`data_p5 / data_p95` = the range the plant really ran. Where a side has no evidence (e.g. no over-cooking
limit, because Viscosity-low fails are rare), **stay inside the data range** - the data cannot speak beyond it.

In [ ]:
win = golden.window(results, df)
win.round(3)

### Combined window check
Rules from `config.GOLDEN_WINDOW_RULES`. A batch is **inside** when it meets every rule.
This is the test of the window itself: fail rate inside vs outside, overall and **month by month**.

In [ ]:
print("Rules:", config.GOLDEN_WINDOW_RULES)
for event in ["fail", "kappa_high", "visc_high", "visc_low"]:
    s, pm = golden.window_check(df, config.GOLDEN_WINDOW_RULES, event)
    print(f"{golden.OUTCOMES[event]:15s} inside {s['n_inside']} ({s['share_inside']:.0%}) | "
          f"fail inside {s['fail_inside']:.0%} {tuple(round(v, 2) for v in s['fail_inside_ci'])} vs "
          f"outside {s['fail_outside']:.0%} {tuple(round(v, 2) for v in s['fail_outside_ci'])} | "
          f"Fisher p {s['fisher_p']:.1e} | MH RR outside {s['mh_rr_outside']:.2f} (p {s['mh_p']:.3f}) | months {s['months_agree']}")

s, pm = golden.window_check(df, config.GOLDEN_WINDOW_RULES, "fail")
plots.window_months(pm, "Fail QC: inside vs outside the window, by month");
pm.round(2)

## Reading the result
- **Grade A** -> propose a trial / SOP candidate. **B** -> trial to confirm. **C / -** -> do not adjust on this evidence.
- A window says *where the fail rate is lower in history*, not that moving X **causes** it. Check the chemistry, then confirm with a trial (e.g. same week, alternate the setting).
- Viscosity-low fails are rare, so the data has **no upper limit** for cooking strength: never push X beyond `data_p95`.
- Log each run in `update.md`.

## Scatter - P-factor overshoot vs Viscosity
Each dot = one batch. Blue band = Viscosity spec. Dashed line = cut from the screen. Change `X_COL` to look at another X.

In [ ]:
import matplotlib.pyplot as plt

X_COL, X_LABEL = "pfactor_overshoot", "P-factor overshoot (final - setpoint)"
CUT = 64.3                                          # cut from the golden screen
v_lo, v_hi = config.GOLDEN_SPEC["press1_viscosity"]

d = df[df[X_COL] <= df[X_COL].quantile(0.99)]       # hide the top 1% (up to ~600) so the plot is readable
fig, ax = plt.subplots(figsize=(8, 5))
for flag, name, color in [(0, "Non-Golden", "#eb6834"), (1, "Golden", "#2a78d6")]:
    s = d[d["golden"] == flag]
    ax.scatter(s[X_COL], s["press1_viscosity"], s=18, alpha=0.6, color=color, label=f"{name} (n={len(s)})")
ax.axhspan(v_lo, v_hi, color="#cde2fb", alpha=0.35, lw=0, label=f"Viscosity spec {v_lo}-{v_hi}")
ax.axvline(CUT, color="#0b0b0b", ls="--", lw=1)
ax.text(CUT, ax.get_ylim()[1], f" cut {CUT}", va="top")
ax.set_xlabel(X_LABEL)
ax.set_ylabel("Press 1 Viscosity")
ax.set_title(f"{X_LABEL} vs Viscosity, coloured by Golden / Non-Golden", loc="left")
ax.legend(frameon=False, loc="upper right")
plt.show()
print(f"Spearman overshoot vs Viscosity: {df[[X_COL, 'press1_viscosity']].corr(method='spearman').iloc[0, 1]:+.2f}")